# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [41]:
!pip install -q duckdb pandas pyarrow huggingface_hub

from huggingface_hub import login

login()

from huggingface_hub import whoami

whoami()

{'type': 'user',
 'id': '68b994300b1b9b7236e18e88',
 'name': 'wasif332',
 'fullname': 'Wasif',
 'email': 'wasifbhatti889@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1785542400,
 'isPro': False,
 'avatarUrl': '/avatars/9376307be978258f3f90474c5cfc61ff.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Flyrank',
   'role': 'read',
   'createdAt': '2026-07-28T16:12:45.954Z'}}}

In [42]:
import os
from huggingface_hub import get_token

token = get_token()
print(token[:10])

hf_MGCotQP


In [53]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

df = pd.DataFrame(ds)

df["report_date"] = pd.to_datetime(df["report_date"])

df.head()

import pandas as pd

rows = []

for row in ds:
    rows.append(row)
    if len(rows) == 1000:
        break

df = pd.DataFrame(rows)

df.head()

df.columns.tolist()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

KeyboardInterrupt: 

1. Unit of analysis + time window
One row represents the daily performance record of one content page for one client.

The time window used for analysis is March 2026 (2026-03), a mid-panel month selected for development.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions


## Label / Proxy

Refresh opportunity ranking based on content performance signals.


## Context

- report_date
- client_hash_id
- content_hash_id


## Excluded

Future outcome fields and any label-derived fields.

Reason:
These fields would not be available at the decision moment and may introduce data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [46]:
df["report_date"] = pd.to_datetime(df["report_date"])

march_df = df[
    (df["report_date"] >= "2026-03-01") &
    (df["report_date"] < "2026-04-01")
]

march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,leak_feature


In [47]:
march_df[
[
"report_date",
"client_hash_id",
"content_hash_id"
]
].head()

,report_date,client_hash_id,content_hash_id


In [48]:
print("Rows:", len(march_df))

print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

Rows: 0
Date range: NaT to NaT


In [49]:
available_rows = march_df[
    march_df["gsc_data_available"] == True
].shape[0]

available_rows

0

In [50]:
feature_frame = march_df[
[
"gsc_impressions",
"gsc_clicks",
"gsc_avg_position",
"ga4_pageviews",
"ga4_engaged_sessions"
]
]

feature_frame.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions


## Feature Availability

gsc_impressions:
Available because historical search impressions are recorded before making decisions.

gsc_clicks:
Available because previous search clicks are already collected.

gsc_avg_position:
Available because previous ranking positions are measured.

ga4_pageviews:
Available because analytics traffic data is already available.

ga4_engaged_sessions:
Available because previous user engagement is recorded.

The leakage feature directly copied information from the outcome.

This creates an unrealistically good result because the model receives information that would not be available at prediction time.

The feature was removed to keep the evaluation honest.

In [51]:
march_df["leak_feature"] = march_df["ga4_pageviews"]

march_df[
[
"leak_feature",
"ga4_pageviews"
]
].head()

,leak_feature,ga4_pageviews


## Leakage Lesson

The leakage feature copied information directly from another performance outcome.

This creates an unrealistic advantage because the information would not be available at the decision moment.

The feature is removed to keep the analysis honest.

In [52]:
march_df.drop(
columns=["leak_feature"],
inplace=True
)

/tmp/ipykernel_9194/3875630026.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  march_df.drop(


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

A limitation of this dataset is that it contains performance signals but does not include direct information about content quality or confirmed refresh outcomes.

Another limitation is that GSC and GA4 availability may differ between clients, creating uneven historical coverage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.